In [1]:

import gc
import json
import sys
from collections import Counter
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocessing.pipeline import (
    stratified_split,
    fit_scaler,
    apply_scaler,
    smote_train_only,
)

LABEL = "Attack Type"
NORMAL = "Normal Traffic"
SEED = 42

df = pd.read_parquet(
    ROOT / "data" / "interim" / "cleaned.parquet"
)

schema = json.loads(
    (ROOT / "data" / "schema" / "feature_schema.json").read_text(
        encoding="utf-8"
    )
)

FEATURE_COLUMNS = [
    feature["name"]
    for feature in schema["features"]
]

DROP_FEATURES = {
    "Active Mean",
    "Active Max",
    "Active Min",
    "Idle Mean",
    "Idle Max",
    "Idle Min",
}

assert len(FEATURE_COLUMNS) == 46
assert "Destination Port" in FEATURE_COLUMNS
assert DROP_FEATURES.isdisjoint(FEATURE_COLUMNS)

X = df[FEATURE_COLUMNS].copy()
y = df[LABEL]

print("Selected features:", X.shape[1])
print("Dataset shape:", X.shape)
print("Destination Port retained:", "Destination Port" in X.columns)
print(X.shape, y.shape)

del df
gc.collect()


Selected features: 46
Dataset shape: (2520590, 46)
Destination Port retained: True
(2520590, 46) (2520590,)


0

In [2]:

le = LabelEncoder()

y_enc = pd.Series(
    le.fit_transform(y),
    index=y.index,
    name="label"
)

mapping = {
    int(i): str(name)
    for i, name in enumerate(le.classes_)
}

print(mapping)

(ROOT / "models").mkdir(exist_ok=True)

joblib.dump(
    le,
    ROOT / "models" / "label_encoder.pkl"
)


import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocessing.pipeline import (
    stratified_split,
    fit_scaler,
    apply_scaler,
    smote_train_only,
)



{0: 'Bots', 1: 'Brute Force', 2: 'DDoS', 3: 'DoS', 4: 'Normal Traffic', 5: 'Port Scanning', 6: 'Web Attacks'}


In [3]:

X_train, X_val, X_test, y_train, y_val, y_test = stratified_split(
    X, y_enc, seed=SEED
)

for name, part in [
    ("train", y_train),
    ("val", y_val),
    ("test", y_test),
]:
    share = part.value_counts(normalize=True).sort_index().round(4)
    print(name, len(part), dict(share))

assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)

print("Split checks passed")


for name, part in [
    ("train", y_train),
    ("val", y_val),
    ("test", y_test)
]:
    share = part.value_counts(normalize=True).sort_index().round(4)
    print(name, len(part), dict(share))

assert not (set(X_train.index) & set(X_val.index))
assert not (set(X_train.index) & set(X_test.index))
assert not (set(X_val.index) & set(X_test.index))

print("Split checks passed")


train 1764413 {0: np.float64(0.0008), 1: np.float64(0.0036), 2: np.float64(0.0508), 3: np.float64(0.0769), 4: np.float64(0.8311), 5: np.float64(0.036), 6: np.float64(0.0009)}
val 378088 {0: np.float64(0.0008), 1: np.float64(0.0036), 2: np.float64(0.0508), 3: np.float64(0.0769), 4: np.float64(0.8311), 5: np.float64(0.036), 6: np.float64(0.0009)}
test 378089 {0: np.float64(0.0008), 1: np.float64(0.0036), 2: np.float64(0.0508), 3: np.float64(0.0769), 4: np.float64(0.8311), 5: np.float64(0.036), 6: np.float64(0.0008)}
Split checks passed
train 1764413 {0: np.float64(0.0008), 1: np.float64(0.0036), 2: np.float64(0.0508), 3: np.float64(0.0769), 4: np.float64(0.8311), 5: np.float64(0.036), 6: np.float64(0.0009)}
val 378088 {0: np.float64(0.0008), 1: np.float64(0.0036), 2: np.float64(0.0508), 3: np.float64(0.0769), 4: np.float64(0.8311), 5: np.float64(0.036), 6: np.float64(0.0009)}
test 378089 {0: np.float64(0.0008), 1: np.float64(0.0036), 2: np.float64(0.0508), 3: np.float64(0.0769), 4: np.fl

In [4]:

INTERIM = ROOT / "data" / "interim"

for name, Xp, yp in [
    ("train", X_train, y_train),
    ("val", X_val, y_val),
    ("test", X_test, y_test)
]:
    out = Xp.copy()
    out["label"] = yp
    out.to_parquet(
        INTERIM / f"split_{name}_unscaled.parquet",
        index=False
    )
    print(f"Saved split_{name}_unscaled.parquet: {out.shape}")

del X, y, y_enc
gc.collect()


Saved split_train_unscaled.parquet: (1764413, 47)
Saved split_val_unscaled.parquet: (378088, 47)
Saved split_test_unscaled.parquet: (378089, 47)


20

In [5]:

schema = json.loads(
    (ROOT / "data" / "schema" / "feature_schema.json").read_text(
        encoding="utf-8"
    )
)

schema_cols = [f["name"] for f in schema["features"]]

assert list(X_train.columns) == schema_cols, (
    "Columns differ from feature_schema.json"
)

print("Columns match feature_schema.json")
print("Feature count:", len(schema_cols))


Columns match feature_schema.json
Feature count: 46


In [6]:

scaler = fit_scaler(X_train)

X_train_s = apply_scaler(scaler, X_train)
X_val_s = apply_scaler(scaler, X_val)
X_test_s = apply_scaler(scaler, X_test)

joblib.dump(scaler, ROOT / "models" / "scaler.pkl")

print(
    "Largest train column average (should be about 0):",
    X_train_s.mean().abs().max()
)

print("Scaler saved successfully")

del X_train, X_val, X_test
gc.collect()


Largest train column average (should be about 0): 6.623736572889753e-17
Scaler saved successfully


0

In [7]:

train_counts = Counter(y_train)

print("Train class counts:", dict(sorted(train_counts.items())))

FLOOR = 10000

strategy = {
    cls: FLOOR
    for cls, count in train_counts.items()
    if count < FLOOR
}

print("Classes raised to", FLOOR, ":", strategy)

X_res, y_res = smote_train_only(
    X_train_s,
    y_train,
    floor=FLOOR,
    seed=SEED
)

print("After SMOTE:", dict(sorted(Counter(y_res).items())))

print(
    "Train rows",
    len(X_train_s),
    "->",
    len(X_res),
    "| val and test untouched:",
    len(X_val_s),
    len(X_test_s)
)


Train class counts: {0: 1364, 1: 6405, 2: 89610, 3: 135621, 4: 1466427, 5: 63486, 6: 1500}
Classes raised to 10000 : {6: 10000, 0: 10000, 1: 10000}
After SMOTE: {0: 10000, 1: 10000, 2: 89610, 3: 135621, 4: 1466427, 5: 63486, 6: 10000}
Train rows 1764413 -> 1785144 | val and test untouched: 378088 378089


In [8]:

PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

def save(Xp, yp, name):
    out = Xp.reset_index(drop=True).copy()
    out["label"] = np.asarray(yp)
    out.to_parquet(PROC / name, index=False)
    print(name, out.shape)

save(X_train_s, y_train, "train_scaled.parquet")
save(X_res, y_res, f"train_smote_floor{FLOOR}.parquet")
save(X_val_s, y_val, "val_scaled.parquet")
save(X_test_s, y_test, "test_scaled.parquet")

del X_train_s, X_val_s, X_test_s, X_res
gc.collect()


train_scaled.parquet (1764413, 47)
train_smote_floor10000.parquet (1785144, 47)
val_scaled.parquet (378088, 47)
test_scaled.parquet (378089, 47)


0